# RAG Pipeline — Financial Document Summarization
**Intern Project | Summarization & Contextualization Team**

### Pipeline Overview
```
PDF Input
  → Extract (text + tables + headers + footers)
  → Chunk (by section, keep tables whole)
  → Embed (all-MiniLM-L6-v2, local, free)
  → Store (ChromaDB, local)
  → Retrieve (cosine similarity)
  → Answer (Gemini)
  → Multi-viewpoint Summarization
  → Evaluation (RAGAS)
```

## Cell 1 — Install Dependencies

In [2]:
!pip install pymupdf pdfplumber sentence-transformers chromadb google-generativeai ragas langchain langchain-community datasets

  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached langchain-1.3.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached tiktoken-0.13.0-cp311-cp311-win_amd64.whl.metadata (6.8 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached openai-2.38.0-py3-none-any.whl.metadata (31 kB)
  Using cached instructor-1.15.1-py3-none-any.whl.metadata (12 kB)
  Using cached scikit_network-0.33.5-cp311-cp311-win_amd64.whl.metadata (4.6 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langchain_openai-1.2.2-py3-none-any.whl.metadata (3.1 kB)
  Using cached langgraph-1.2.1-py3-none-any.whl.metadata (8.0 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'd:\\V3_Internship\\Code\\RAG\\.venv\\Lib\\site-packages\\sqlalchemy\\ext\\asyncio\\exc.py'
Check the permissions.



## Cell 2 — Imports & Config

In [ ]:
import fitz  # PyMuPDF
import pdfplumber
import json
import re
import shutil
import os
import google.generativeai as genai
import chromadb
from sentence_transformers import SentenceTransformer

# ─── CONFIG ───────────────────────────────────────────────
#GEMINI_API_KEY = ""   # ← paste your key here
PDF_PATH       = "data/finance_evaluation.pdf" # ← your PDF filename
CHROMA_PATH    = "./chroma_store"
COLLECTION     = "financial_report"
CHUNK_SIZE     = 200   # words per chunk
CHUNK_OVERLAP  = 30
TOP_K          = 3     # chunks to retrieve
# ──────────────────────────────────────────────────────────

genai.configure(api_key=GEMINI_API_KEY)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
# gemini = genai.GenerativeModel("gemini-2.0-flash")

import requests

def ask_llm(prompt):
    response = requests.post("http://localhost:11434/api/generate", json={
        "model": "gemma3:1b",
        "prompt": prompt,
        "stream": False
    })
    return response.json()["response"]

print("✅ Imports done")

d:\V3_Internship\Code\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Yugesh\AppData\Local\Temp\ipykernel_1396\3670095693.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7424.31it/s]


✅ Imports done


## Cell 3 — PDF Extraction

In [2]:
import fitz
import pdfplumber

FOOTER_MARGIN = 50
HEADER_MARGIN = 70

def extract_text(path):
    doc = fitz.open(path)

    header_content = []
    footer_content = []
    main_content = ""
    all_tables_json = []
    image_count = 0

    with pdfplumber.open(path) as pdf:
        for page_num in range(len(doc)):
            page = doc[page_num]
            plumber_page = pdf.pages[page_num]

            page_height = page.rect.height
            footer_y_start = page_height - FOOTER_MARGIN
            header_y_end = HEADER_MARGIN

            blocks = page.get_text("blocks")
            blocks = sorted(blocks, key=lambda b: (b[1], b[0]))

            tables = plumber_page.find_tables()
            table_regions = []

            # Extract tables
            for table in tables:
                table_regions.append(table.bbox)

                extracted = table.extract()

                if extracted and len(extracted) > 1:
                    headers = [
                        h.replace("\n", " ").strip() if h else f"column_{i}"
                        for i, h in enumerate(extracted[0])
                    ]

                    table_json = []

                    for row in extracted[1:]:
                        cleaned_row = [
                            cell.replace("\n", " ").strip() if cell else ""
                            for cell in row
                        ]

                        row_dict = dict(zip(headers, cleaned_row))
                        table_json.append(row_dict)

                    all_tables_json.append({
                        "page": page_num + 1,
                        "table_data": table_json
                    })

            # Extract text blocks
            for block in blocks:
                x0, y0, x1, y1, text = block[:5]
                text = text.strip()

                if not text:
                    continue

                # HEADER JSON
                if y1 <= header_y_end:
                    header_content.append({
                        "page": page_num + 1,
                        "text": text
                    })
                    continue

                # FOOTER JSON
                if y0 >= footer_y_start:
                    footer_content.append({
                        "page": page_num + 1,
                        "text": text
                    })
                    continue

                # Skip table text from PyMuPDF
                inside_table = False
                for tx0, ty0, tx1, ty1 in table_regions:
                    if y0 >= ty0 and y1 <= ty1:
                        inside_table = True
                        break

                if inside_table:
                    continue

                main_content += text + "\n\n"

            image_count += len(page.get_images())

    return header_content, main_content, footer_content, all_tables_json, image_count


# Run
path = "data/finance_evaluation.pdf"

header_content, main_content, footer_content, tables_json, image_count = extract_text(path)

## Cell 4 — Chunking

In [3]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Word-based chunking with overlap."""
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


def split_by_sections(text):
    """Split text on SECTION headings so each chunk stays topically focused."""
    parts   = re.split(r'(SECTION\s+\d+[:\s][^\n]*)', text)
    results = []
    current = "INTRODUCTION"
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if re.match(r'SECTION\s+\d+', part):
            current = part
        else:
            results.append({"section": current, "text": part})
    return results


def table_to_readable(table_data):
    """Convert table rows (list of dicts) to natural-language text for embedding."""
    if not table_data or not isinstance(table_data, list):
        return json.dumps(table_data)
    headers = list(table_data[0].keys()) if isinstance(table_data[0], dict) else []
    lines   = ["Columns: " + " | ".join(headers)] if headers else []
    for row in table_data:
        lines.append(" | ".join(f"{k}: {v}" for k, v in row.items()))
    return "\n".join(lines)


def prepare_chunks(main_content, header_content, footer_content, tables_json):
    documents    = []
    header_text  = " | ".join(h["text"] for h in header_content) if header_content else ""

    # 1. Main content — split by section, then chunk each section
    sections = split_by_sections(main_content)
    chunk_id = 0
    for section in sections:
        for chunk in chunk_text(section["text"]):
            documents.append({
                "type"     : "main_content",
                "chunk_id" : chunk_id,
                "text"     : chunk,
                "metadata" : {
                    "chunk_index"     : chunk_id,
                    "section"         : section["section"],
                    "document_header" : header_text,
                }
            })
            chunk_id += 1

    # 2. Tables — each table is one chunk, never split
    for idx, table in enumerate(tables_json):
        documents.append({
            "type"     : "table",
            "chunk_id" : f"table_{idx}",
            "text"     : table_to_readable(table["table_data"]),
            "raw"      : table["table_data"],   # kept for precise LLM use
            "metadata" : {
                "table_index"     : idx,
                "page"            : table["page"],
                "document_header" : header_text,
            }
        })

    return documents


documents = prepare_chunks(main_content, header_content, footer_content, tables_json)

for t in ["main_content", "table"]:
    print(f"{t}: {sum(1 for d in documents if d['type'] == t)} chunks")
print(f"\nSample main chunk:\n{documents[0]['text'][:300]}")

main_content: 7 chunks
table: 1 chunks

Sample main chunk:
DOCUMENT HEADER: QUARTERLY FINANCIAL REVIEW ISSUER: APEX GLOBAL WEALTH MANAGEMENT GROUP REPORTING PERIOD: Q1 2026 TARGET AUDIENCE: PRIVATE WEALTH PORTFOLIO CLIENTS


In [4]:
# See exactly what your extracted text looks like
print(main_content[:1000])

DOCUMENT HEADER: QUARTERLY FINANCIAL REVIEW

ISSUER: APEX GLOBAL WEALTH MANAGEMENT GROUP

REPORTING PERIOD: Q1 2026

TARGET AUDIENCE: PRIVATE WEALTH PORTFOLIO CLIENTS

SECTION 1: MONETARY POLICY AND CAPITAL MARKETS CONTEXT

The first quarter of 2026 has demonstrated remarkable resilience across global capital markets, characterized by a

transition toward normalizing monetary policies and steady corporate earnings growth. Central banks globally have

initiated a measured approach to interest rate adjustments, easing structural pressures on both fixed income and

equity valuations. While macroeconomic uncertainties linger regarding sticky supply-chain components, consumer

spending metrics remain healthy, supporting a baseline expansion model. In this dynamic environment, our core

asset allocation strategy prioritized high-quality, dividend-yielding equities paired with opportunistic fixed-income

durations. This dual-pronged focus allowed our managed portfolios to capture equity upsid

## Cell 5 — Embed & Store in ChromaDB

In [6]:
# Cell — Force close and reopen
import gc

# Close existing client
try:
    chroma_client.reset()
except:
    pass

chroma_client = None
collection = None
gc.collect()

# Reopen fresh
import chromadb
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ Reconnected | chunks in DB: {collection.count()}")

✅ Reconnected | chunks in DB: 8


In [7]:
# ── Wipe old store ────────────────────────────────────────
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print("🗑️  Old ChromaDB deleted")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(
    name     = COLLECTION,
    metadata = {"hnsw:space": "cosine"}
)


def store_chunks(documents):
    ids, embeddings, texts, metadatas = [], [], [], []

    for doc in documents:
        chunk_id  = f"{doc['type']}_{doc['chunk_id']}"
        embedding = embed_model.encode(doc["text"]).tolist()

        meta = {"type": doc["type"]}
        if "metadata" in doc:
            for k, v in doc["metadata"].items():
                if isinstance(v, (str, int, float)):
                    meta[k] = str(v)

        ids.append(chunk_id)
        embeddings.append(embedding)
        texts.append(doc["text"])
        metadatas.append(meta)
        print(f"  Embedded: {chunk_id}")

    collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
    print(f"\n✅ Stored {len(ids)} chunks in ChromaDB")


store_chunks(documents)
print(f"Total in DB: {collection.count()}")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: './chroma_store\\bc31bcb9-4541-4348-ac3a-339b664460a1\\data_level0.bin'

## Cell 6 — Retrieval

In [8]:
def retrieve(query, top_k=TOP_K):
    """Embed query and fetch top-k most similar chunks (main + table only)."""
    query_embedding = embed_model.encode(query).tolist()
    results = collection.query(
        query_embeddings = [query_embedding],
        n_results        = top_k,
        include          = ["documents", "metadatas", "distances"],
        where            = {"type": {"$in": ["main_content", "table"]}}
    )
    return results


# ── Sanity check ─────────────────────────────────────────
test_queries = [
    "What is the VaR confidence interval for the portfolio?",
    "What is the current allocation percentage for Emerging Markets Equities?",
    "What strategic actions are planned for over-allocated assets?"
]

for query in test_queries:
    results = retrieve(query)
    print(f"\nQ: {query}")
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        sim = 1 - dist
        print(f"  [{meta['type']}] similarity: {sim:.3f} | section: {meta.get('section','table')} | {doc[:80]}...")


Q: What is the VaR confidence interval for the portfolio?
  [main_content] similarity: 0.405 | section: SECTION 5: RISK MANAGEMENT & PORTFOLIO VOLATILITY MODELING | Our ongoing quantitative stress-testing frameworks simulate portfolio resilience...
  [main_content] similarity: 0.294 | section: INTRODUCTION | DOCUMENT HEADER: QUARTERLY FINANCIAL REVIEW ISSUER: APEX GLOBAL WEALTH MANAGEMEN...
  [main_content] similarity: 0.255 | section: SECTION 6: FORWARD OUTLOOK AND STRATEGIC ACTION PLAN | As we progress through the subsequent quarters of fiscal year 2026, our overarch...

Q: What is the current allocation percentage for Emerging Markets Equities?
  [main_content] similarity: 0.489 | section: SECTION 4: EQUITY SECTOR DRILLDOWN & VELOCITY TRENDS | The equity component of our client accounts achieved outsized results via system...
  [main_content] similarity: 0.475 | section: SECTION 1: MONETARY POLICY AND CAPITAL MARKETS CONTEXT | The first quarter of 2026 has demonstrated remarkable r

## Cell 7 — RAG Answer Generation

In [9]:
def ask(question):
    results = retrieve(question)
    chunks  = results["documents"][0]
    context = "\n\n---\n\n".join(chunks)

    prompt = f"""You are a financial document analyst.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say: "Not found in document."
Do not use any outside knowledge.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""

    answer = ask_llm(prompt)
    return answer, chunks


# Test
answer, chunks = ask("What is the VaR confidence interval for the portfolio?")
print(answer)


# Test
answer, used_chunks = ask("What is the VaR confidence interval and Beta benchmark for the portfolio?")
print("ANSWER:\n", answer)
print("\nCHUNKS USED:")
for c in used_chunks:
    print(" -", c[:120], "...")

95%
ANSWER:
 95%
0.88


CHUNKS USED:
 - Our ongoing quantitative stress-testing frameworks simulate portfolio resilience against sudden macroeconomic shifts, su ...
 - DOCUMENT HEADER: QUARTERLY FINANCIAL REVIEW ISSUER: APEX GLOBAL WEALTH MANAGEMENT GROUP REPORTING PERIOD: Q1 2026 TARGET ...
 - The first quarter of 2026 has demonstrated remarkable resilience across global capital markets, characterized by a trans ...


## Cell 8 — Multi-Viewpoint Summarization

In [11]:
VIEWPOINTS = {
    "Investor" : (
        "Summarize this quarterly financial report from an investor's perspective. "
        "Focus on portfolio returns, asset performance, and growth opportunities."
    ),
    "Compliance Officer" : (
        "Summarize this quarterly financial report from a compliance officer's perspective. "
        "Focus on risk controls, regulatory posture, VaR limits, and conservative guidelines."
    ),
    "C-Suite Executive" : (
        "Summarize this quarterly financial report from a C-suite executive's perspective. "
        "Focus on strategic decisions, rebalancing actions, and forward outlook."
    ),
    "Risk Manager" : (
        "Summarize this quarterly financial report from a risk manager's perspective. "
        "Focus on volatility metrics, stress testing, downside risks, and defensive positions."
    ),
}

def summarize_viewpoints(viewpoints):
    results = retrieve("quarterly financial portfolio performance risk strategy outlook", top_k=5)
    chunks  = results["documents"][0]
    context = "\n\n---\n\n".join(chunks)

    summaries = {}
    for viewpoint, instruction in viewpoints.items():
        prompt = f"""{instruction}
Use ONLY the information in the context below. Do not add outside knowledge.
Keep the summary to 3-5 sentences.

CONTEXT:
{context}

SUMMARY:"""

        summaries[viewpoint] = ask_llm(prompt)
        print(f"\n{'='*60}")
        print(f"  {viewpoint.upper()} VIEW")
        print(f"{'='*60}")
        print(summaries[viewpoint])

    return summaries, chunks

summaries, summary_chunks = summarize_viewpoints(VIEWPOINTS)


  INVESTOR VIEW
Here’s a summary of the quarterly financial report from an investor’s perspective, focusing on portfolio returns, asset performance, and growth opportunities, based solely on the provided text:

**Portfolio Returns:** The portfolio is currently performing well, with profits from cyclical equities being re-routed into defensive value plays and sovereign fixed-income, resulting in attractive nominal yields.

**Asset Performance:** The report indicates a focus on high-quality, dividend-yielding equities paired with opportunistic fixed-income, which has generated positive returns.

**Growth Opportunities:** The report suggests continued focus on phased profit-taking within cyclical equities, while maintaining a liquidity reserve outside active investment horizons to mitigate potential losses during market downturns. The team is actively adjusting individual wealth criteria based on market conditions.

  COMPLIANCE OFFICER VIEW
As a compliance officer, the report emphasizes

## Cell 9 — Evaluation with RAGAS

Evaluates:
- **Faithfulness** — Does the answer only use info from retrieved chunks?
- **Answer Relevancy** — Does the answer address the question?
- **Context Recall** — Were all relevant chunks retrieved?
- **Context Precision** — Were retrieved chunks actually useful?

In [15]:
!pip install langchain-ollama

In [16]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from datasets import Dataset

# ── Point RAGAS to Ollama ─────────────────────────────────
judge_llm   = LangchainLLMWrapper(OllamaLLM(model="gemma3:1b"))
judge_embed = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="gemma3:1b"))

# ── Build eval dataset (same as before) ──────────────────
eval_questions = [
    {
        "question"     : "What is the VaR confidence interval for the portfolio?",
        "ground_truth" : "The VaR parameters track a 95% confidence interval that potential weekly downside variance will not exceed 2.1%."
    },
    {
        "question"     : "What is the current allocation for Emerging Markets Equities?",
        "ground_truth" : "The current allocation for Emerging Markets Equities is 4.1%, below the 5.0% target, with a Q1 return of -2.4%."
    },
    {
        "question"     : "What strategic actions are planned for Domestic Large-Cap Equities?",
        "ground_truth" : "Domestic Large-Cap Equities are over-allocated at 32.4% vs the 30.0% target and the strategic action is to Trim to Target."
    },
    {
        "question"     : "What is the Beta benchmark for the portfolio?",
        "ground_truth" : "Volatility metrics are calibrated to a Beta of 0.88 against the broader market index."
    },
]

eval_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

for item in eval_questions:
    answer, chunks = ask(item["question"])
    eval_data["question"].append(item["question"])
    eval_data["answer"].append(answer)
    eval_data["contexts"].append(chunks)
    eval_data["ground_truth"].append(item["ground_truth"])
    print(f"✅ {item['question'][:60]}...")

dataset = Dataset.from_dict(eval_data)

# ── Run RAGAS ─────────────────────────────────────────────
results = evaluate(
    dataset,
    metrics    = [faithfulness, answer_relevancy, context_recall, context_precision],
    llm        = judge_llm,
    embeddings = judge_embed,
)

results_df = results.to_pandas()
print(results_df[[
    "question", "faithfulness", "answer_relevancy",
    "context_recall", "context_precision"
]].to_string())
print("\nMean Scores:")
print(results_df[[
    "faithfulness", "answer_relevancy",
    "context_recall", "context_precision"
]].mean())

C:\Users\Yugesh\AppData\Local\Temp\ipykernel_1396\3350717966.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
C:\Users\Yugesh\AppData\Local\Temp\ipykernel_1396\3350717966.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
C:\Users\Yugesh\AppData\Local\Temp\ipykernel_1396\3350717966.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

✅ What is the VaR confidence interval for the portfolio?...
✅ What is the current allocation for Emerging Markets Equities...
✅ What strategic actions are planned for Domestic Large-Cap Eq...
✅ What is the Beta benchmark for the portfolio?...


Evaluating:  56%|█████▋    | 9/16 [00:42<00:20,  2.97s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt context_recall_classification_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[2]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating: 100%|██████████| 16/16 [00:43<00:00,  2.75s/it]


KeyError: "['question'] not in index"

## Cell 10 — Interpreting Your Scores

| Metric | What it means | Good score |
|---|---|---|
| **Faithfulness** | Answer only uses retrieved context, no hallucination | > 0.8 |
| **Answer Relevancy** | Answer actually addresses the question | > 0.8 |
| **Context Recall** | Retrieved chunks covered the right information | > 0.7 |
| **Context Precision** | Retrieved chunks were all useful (no noise) | > 0.7 |

### If scores are LOW — what to fix:
```
Low Faithfulness      → LLM is going off-script. Strengthen your prompt:
                         add "Do not use outside knowledge" more explicitly.

Low Answer Relevancy  → Retrieved chunks don't match the question well.
                         Try smaller chunk_size (100-150 words).

Low Context Recall    → Right chunks not being retrieved.
                         Try larger top_k (5 instead of 3).

Low Context Precision → Too many irrelevant chunks retrieved.
                         Try smaller top_k or stricter filtering.
```

## Cell 11 — Full Pipeline Runner (One Shot)
Run this cell to execute the entire pipeline end-to-end.

In [ ]:
print("🚀 Running full RAG pipeline...\n")

# Step 1: Extract
print("[1/5] Extracting PDF...")
main_content, header_content, footer_content, tables_json = extract_pdf(PDF_PATH)

# Step 2: Chunk
print("[2/5] Chunking...")
documents = prepare_chunks(main_content, header_content, footer_content, tables_json)
print(f"      {sum(1 for d in documents if d['type']=='main_content')} main chunks | {sum(1 for d in documents if d['type']=='table')} table chunks")

# Step 3: Embed + Store
print("[3/5] Embedding and storing...")
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})
store_chunks(documents)

# Step 4: Multi-viewpoint summarization
print("\n[4/5] Generating multi-viewpoint summaries...")
summaries, _ = summarize_viewpoints(VIEWPOINTS)

# Step 5: Q&A demo
print("\n[5/5] Demo Q&A...")
demo_q = "What is the overall portfolio return for Q1 2026 and what are the key strategic actions?"
answer, _ = ask(demo_q)
print(f"Q: {demo_q}")
print(f"A: {answer}")

print("\n✅ Pipeline complete! Run Cell 9 for RAGAS evaluation.")